# Volume 9, Chapter 7: Group Cohomology (Python companion)

Carry-lookahead adders and the carry cocycle, all extensions of the Klein four-group by Z/2, and Hilbert's Theorem 90 as an algorithm in finite fields.

*Before each solution, predict what the output will look like.*

## Exercises

**P7.1 (★★) CS application: carry-lookahead adders and the cocycle identity.** A ripple-carry adder adds two $n$-bit numbers digit by digit, passing the carry along: $n$ sequential steps. Fast hardware adders (carry-lookahead, Kogge–Stone) compute all carries in $O(\log n)$ steps. Encode the effect of digit position $i$ on an incoming carry as a pair $(g_i,p_i)$: *generate* ($x_i+y_i\ge b$) and *propagate* ($x_i+y_i=b-1$), and combine positions by $(g,p)\circ(g',p')=(g\lor(p\land g'),\,p\land p')$. (a) Check that $\circ$ is associative, so that the carries are a *prefix scan* that can be computed in parallel in $\lceil\log_2n\rceil$ rounds. (b) Implement the Kogge–Stone scan for base $2$ and base $10$, compare with ordinary addition on random inputs, and count the rounds. (c) Check numerically that the carry $c(x,y)$ of Example 7.9 satisfies the cocycle identity, and explain the link: the associativity of $\circ$ is the associativity of addition, which is what the cocycle identity encodes (proof of Theorem 7.7).

**P7.2 (★★) All extensions of $(\Z/2)^2$ by $\Z/2$.** With trivial action, enumerate the normalised $2$-cocycles $f\colon G\times G\to\Z/2$ for $G=(\Z/2)^2$, and the coboundaries, and check that $H^2(G,\Z/2)$ has $8$ elements. For each class, build $E_f$ (Theorem 7.7) and identify it by its numbers of elements of orders $1,2,4$: which classes give $(\Z/2)^3$, $\Z/4\times\Z/2$, the dihedral group $D_4$ and the quaternion group $Q_8$, and how many of each?

**P7.3 (★) Hilbert's Theorem 90 in a finite field.** For the cyclic extension $\mathbb F_{p^n}/\mathbb F_p$ with generator $\sigma(x)=x^p$, the norm is $N(x)=x^{1+p+\dots+p^{n-1}}$. (a) Implement $\mathbb F_{p^n}$, pick random $x$ of norm $1$, and find $b$ with $x=b/\sigma(b)$ by the formula of the proof of Theorem 7.4, $b=\sum_if(\sigma^i)\sigma^i(c)$. (b) Count the elements of norm $1$ and compare with the number of values of $b/\sigma(b)$.

## Solutions

### P7.1

In [ ]:
import random, math, itertools

def op(a, b):
    """(g, p) o (g', p'): first the lower block (g', p'), then the higher block (g, p)."""
    g, p = a; g2, p2 = b
    return (g or (p and g2), p and p2)

pairs = list(itertools.product([False, True], repeat=2))
print("o is associative:", all(op(op(a, b), c) == op(a, op(b, c)) for a in pairs for b in pairs for c in pairs))

def digits(x, b, n): return [(x // b**i) % b for i in range(n)]

def kogge_stone_add(x, y, b, n):
    X, Y = digits(x, b, n), digits(y, b, n)
    gp = [(X[i] + Y[i] >= b, X[i] + Y[i] == b - 1) for i in range(n)]
    # prefix[i] = gp[i] o gp[i-1] o ... o gp[0], computed in log2(n) parallel rounds
    pref, d, rounds = gp[:], 1, 0
    while d < n:
        pref = [op(pref[i], pref[i - d]) if i >= d else pref[i] for i in range(n)]
        d *= 2; rounds += 1
    carry_in = [False] + [pref[i][0] for i in range(n - 1)]       # carry into position i
    S = [(X[i] + Y[i] + carry_in[i]) % b for i in range(n)]
    return sum(s * b**i for i, s in enumerate(S)), pref[n - 1][0], rounds

for b, n in [(2, 64), (10, 20), (2, 1000)]:
    ok = True
    for _ in range(200):
        x, y = random.randrange(b**n), random.randrange(b**n)
        s, cout, rounds = kogge_stone_add(x, y, b, n)
        ok &= (s + cout * b**n == x + y)
    print(f"base {b}, {n} digits: correct on 200 random sums: {ok}, rounds = {rounds}, ceil(log2 n) = {math.ceil(math.log2(n))}")

In [ ]:
def carry(x, y, b): return 1 if x + y >= b else 0
for b in (2, 10, 16):
    ok = all((carry(y, z, b) + carry(x, (y + z) % b, b) - carry((x + y) % b, z, b) - carry(x, y, b)) % b == 0
             for x in range(b) for y in range(b) for z in range(b))
    print(f"base {b}: carry satisfies the cocycle identity: {ok}")

The scan needs $6$ rounds for $64$ bits and $10$ for $1000$ bits, instead of $64$ and $1000$ sequential carry steps; this is why every processor uses a parallel-prefix adder. The trick only works because $\circ$ is associative, so that the carries into all positions can be bracketed in any order. And the cocycle identity for the carry, checked in the second cell, is the same associativity seen from the other side: $\Z/b^2$ is a group (addition is associative) exactly because $c(y,z)+c(x,y+z)=c(x,y)+c(x+y,z)$, both sides counting the carries out of the units digit when three digits are added.

### P7.2

In [ ]:
G = [(0, 0), (1, 0), (0, 1), (1, 1)]
def gadd(g, h): return ((g[0] + h[0]) % 2, (g[1] + h[1]) % 2)
nonid = G[1:]
pairs = [(g, h) for g in nonid for h in nonid]

def is_cocycle(f):
    F = lambda g, h: 0 if g == (0, 0) or h == (0, 0) else f[(g, h)]
    return all((F(h, k) - F(gadd(g, h), k) + F(g, gadd(h, k)) - F(g, h)) % 2 == 0 for g in G for h in G for k in G)

cocycles = []
for vals in itertools.product(range(2), repeat=len(pairs)):
    f = dict(zip(pairs, vals))
    if is_cocycle(f): cocycles.append(f)
cob = set()
for bv in itertools.product(range(2), repeat=3):
    b = dict(zip(nonid, bv)); b[(0, 0)] = 0
    cob.add(tuple((b[g] + b[h] - b[gadd(g, h)]) % 2 for g, h in pairs))
print(len(cocycles), "normalised cocycles,", len(cob), "coboundaries, |H^2| =", len(cocycles) // len(cob))

def order_profile(f):
    F = lambda g, h: 0 if g == (0, 0) or h == (0, 0) else f[(g, h)]
    def mul(u, v): return ((u[0] + v[0] + F(u[1], v[1])) % 2, gadd(u[1], v[1]))
    E = [(a, g) for a in range(2) for g in G]; e = (0, (0, 0))
    prof = {1: 0, 2: 0, 4: 0}
    for u in E:
        k, w = 1, u
        while w != e: w = mul(w, u); k += 1
        prof[k] += 1
    abelian = all(mul(u, v) == mul(v, u) for u in E for v in E)
    return prof, abelian

def name(prof, ab):
    if prof[4] == 0: return "(Z/2)^3"
    if ab: return "Z/4 x Z/2"
    return "Q8" if prof[4] == 6 else "D4"

classes = {}
for f in cocycles:
    key = min(tuple((f[p] + c) % 2 for p, c in zip(pairs, cb)) for cb in cob)
    classes.setdefault(key, f)
from collections import Counter
cnt = Counter()
for key, f in classes.items():
    prof, ab = order_profile(f); cnt[name(prof, ab)] += 1
    print(key, prof, "abelian" if ab else "non-abelian", "->", name(prof, ab))
print(cnt)

The $8$ classes give $(\Z/2)^3$ once (the split extension), $\Z/4\times\Z/2$ three times, $D_4$ three times and $Q_8$ once. The asymmetry has a meaning. A class is determined by the map $q(g)=f(g,g)\in\Z/2$, which records whether a lift of $g$ squares to the non-trivial central element; $q$ is a quadratic map on $G=\mathbb F_2^2$ whose associated bilinear form $q(g+h)-q(g)-q(h)$ is the commutator pairing, and there are exactly $2^3=8$ such maps. $Q_8$ is the only one with $q(g)=1$ for all three $g\ne0$ (every element outside the centre has order $4$), which is the cocycle of Exercise 7.3; the three non-zero linear $q$ give $\Z/4\times\Z/2$ (bilinear form $0$: abelian), and the three non-linear $q$ taking the value $1$ exactly once give $D_4$, whose two elements of order $4$ lift that one $g$.

### P7.3

In [ ]:
import sympy as sp
from sympy.polys.galoistools import gf_irreducible_p, gf_mul, gf_rem, gf_pow_mod, gf_add, gf_strip
from sympy.polys.domains import ZZ

class Fq:
    def __init__(self, p, n, seed=1):
        self.p, self.n = p, n
        random.seed(seed)
        while True:
            m = [1] + [random.randrange(p) for _ in range(n)]
            if gf_irreducible_p(m, p, ZZ): break
        self.m = m
    def norm_(self, a):
        a = gf_strip([c % self.p for c in a])
        return gf_rem(a, self.m, self.p, ZZ) if a else []
    def mul(self, a, b): return self.norm_(gf_mul(a, b, self.p, ZZ))
    def add(self, a, b): return gf_add(a, b, self.p, ZZ)
    def pw(self, a, e): return gf_pow_mod(a, e, self.m, self.p, ZZ)
    def inv(self, a): return self.pw(a, self.p**self.n - 2)
    def frob(self, a, i=1): return self.pw(a, self.p**i)
    def rand(self): return self.norm_([random.randrange(self.p) for _ in range(self.n)]) or [1]

def N(F, x): return F.pw(x, sum(F.p**i for i in range(F.n)))

def hilbert90(F, x):
    """x of norm 1: f(sigma^i) = x sigma(x) ... sigma^{i-1}(x); b = sum_i f(sigma^i) sigma^i(c) for a c making b != 0."""
    f = [[1]]
    for i in range(1, F.n): f.append(F.mul(f[-1], F.frob(x, i - 1)))
    while True:
        c = F.rand(); b = []
        for i in range(F.n): b = F.add(b, F.mul(f[i], F.frob(c, i)))
        if b: return b

for p, n in [(2, 8), (3, 5), (101, 3)]:
    F = Fq(p, n)
    ok, tried = True, 0
    while tried < 20:
        x = F.rand()
        if N(F, x) != [1]: continue
        tried += 1
        b = hilbert90(F, x)
        ok &= F.mul(b, F.inv(F.frob(b))) == x
    print(f"F_{p}^{n}: 20 random elements of norm 1 written as b/sigma(b): {ok}")

In [ ]:
p, n = 3, 4
F = Fq(p, n)
elements = [list(c) for c in itertools.product(range(p), repeat=n)]
nonzero = [F.norm_(e) for e in elements if any(e)]
norm_one = sum(1 for x in nonzero if N(F, x) == [1])
quotients = {tuple(F.mul(b, F.inv(F.frob(b)))) for b in nonzero}
print("q =", p**n, ": elements of norm 1:", norm_one, " values of b/sigma(b):", len(quotients), " (q-1)/(p-1) =", (p**n - 1) // (p - 1))

The formula of the proof works as an algorithm. In $\mathbb F_{81}$, the norm map $\mathbb F_{81}^\times\to\mathbb F_3^\times$ is onto, so $80/2=40$ elements have norm $1$; and $b\mapsto b/\sigma(b)=b^{1-p}$ has kernel $\mathbb F_3^\times$, so it takes $40$ values: Theorem 90 says the two sets coincide. In cryptography, the norm-$1$ subgroup of $\mathbb F_{p^n}^\times$ (the *torus*) is used to compress elements in torus-based schemes such as XTR and CEILIDH, precisely because Theorem 90 gives it a rational parametrisation.